# TF-IDF sobre 100 documentos

**Autor:** Jhonn Saeteros  

Este notebook reutiliza el proceso de carga y normalización del archivo `02_booleansearch_Saeteros_Jhonn.ipynb`. Se seleccionan solamente los primeros **100 documentos** del corpus extraído de `txt-files.tar.zip`.

Se calcularán:

- `df(t)`: número de documentos donde aparece el término `t`.
- `idf(t) = log(N / df(t))`: versión clásica sin suavizado.
- `idf(t) = ln(N / df(t) + 1)`: versión solicitada; se colocan explícitamente los paréntesis para evitar ambigüedad.
- Los resultados obtenidos con `TfidfVectorizer` de scikit-learn.

In [1]:
import os
import re
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

RUTA_CORPUS = r"C:\Users\Jhonn22\Downloads\gutenberg_extraido"
NUM_DOCUMENTOS = 100

print("El corpus existe:", os.path.exists(RUTA_CORPUS))


El corpus existe: True


## 1. Obtener los archivos del corpus

La búsqueda recorre las carpetas y conserva únicamente archivos `.txt`. Luego se ordenan los nombres para que la selección de los 100 documentos sea reproducible.

In [2]:
archivos_txt = []

for raiz, _, archivos in os.walk(RUTA_CORPUS):
    for nombre in archivos:
        if nombre.lower().endswith(".txt"):
            archivos_txt.append(os.path.join(raiz, nombre))

archivos_txt.sort()

print("Cantidad total de archivos .txt encontrados:", len(archivos_txt))

if len(archivos_txt) < NUM_DOCUMENTOS:
    raise ValueError(
        f"Se encontraron {len(archivos_txt)} archivos, se necesitan al menos {NUM_DOCUMENTOS}."
    )

archivos_100 = archivos_txt[:NUM_DOCUMENTOS]
print("Cantidad utilizada:", len(archivos_100))
print("Primer documento:", os.path.basename(archivos_100[0]))
print("Último documento:", os.path.basename(archivos_100[-1]))


Cantidad total de archivos .txt encontrados: 78145
Cantidad utilizada: 100
Primer documento: pg10000.txt
Último documento: pg10090.txt


## 2. Normalización y tokenización

Se usa el mismo criterio definido en el notebook booleano:

- Se transforma todo el texto a minúsculas.
- Se eliminan números, puntuación y caracteres especiales.
- Se conservan letras, vocales acentuadas y la letra `ñ`.
- Se separa el texto limpio por espacios.

In [3]:
def normalizar_y_tokenizar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúñü]+", " ", texto)
    return texto.split()

documentos = []
nombres_documentos = []

for ruta in archivos_100:
    with open(ruta, "r", encoding="utf-8", errors="ignore") as archivo:
        documentos.append(archivo.read())
    nombres_documentos.append(os.path.basename(ruta))

documentos_tokenizados = [
    normalizar_y_tokenizar(documento)
    for documento in documentos
]

print("Documentos cargados:", len(documentos))
print("Tokens del primer documento:", documentos_tokenizados[0][:20])
print("Cantidad total de tokens del primer documento:", len(documentos_tokenizados[0]))


Documentos cargados: 100
Tokens del primer documento: ['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'magna', 'carta', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'in', 'the', 'united']
Cantidad total de tokens del primer documento: 17446


## 3. Frecuencia documental `df(t)`

Para calcular `df(t)`, cada documento se convierte en un conjunto de términos con `set()`. De este modo, una palabra se cuenta una sola vez aunque aparezca varias veces dentro del mismo documento.

In [4]:
N = len(documentos_tokenizados)
frecuencia_documental = {}

for tokens in documentos_tokenizados:
    for termino in set(tokens):
        frecuencia_documental[termino] = frecuencia_documental.get(termino, 0) + 1

print("Número de documentos N:", N)
print("Número de términos únicos:", len(frecuencia_documental))
print("Primeros valores df(t):")
pd.DataFrame(
    list(frecuencia_documental.items())[:10],
    columns=["término", "df(t)"]
)


Número de documentos N: 100
Número de términos únicos: 103964
Primeros valores df(t):


,término,df(t)
0,means,100
1,fixed,76
2,arundel,1
3,cities,48
4,needs,63
5,robert,38
6,twenty,88
7,copying,100
8,build,55
9,issues,24


## 4. IDF clásico

Se aplica la fórmula:

\[
idf(t)=\log\left(
rac{N}{df(t)}
ight)
\]

Esta base de `log` puede ser 10 o otra base sin que cambie la selección relativa habitual de los términos; aquí se usa el logaritmo natural, que produce valores equivalentes conceptualmente al registro solicitado.

In [5]:
idf_clasico = {
    termino: math.log(N / df)
    for termino, df in frecuencia_documental.items()
}

terminos_ejemplo = ["the", "and", "book", "project", "gutenberg"]

filas_clasicas = []
for termino in terminos_ejemplo:
    df = frecuencia_documental.get(termino, 0)
    filas_clasicas.append({
        "término": termino,
        "df(t)": df,
        "idf clásico": idf_clasico.get(termino)
    })

pd.DataFrame(filas_clasicas)


,término,df(t),idf clásico
0,the,100,0.000000
1,and,100,0.000000
2,book,86,0.150823
3,project,100,0.000000
4,gutenberg,100,0.000000


## 5. IDF solicitado

Se implementa la expresión indicada con paréntesis explícitos:

\[
idf(t)=\ln\left(
rac{N}{df(t)}+1
ight)
\]

Esta versión evita que el resultado sea negativo cuando el cociente está entre 0 y 1. Para términos presentes en varios documentos, el aumento por `+1` modifica la escala completa.

In [6]:
idf_solicitado = {
    termino: math.log((N / df) + 1)
    for termino, df in frecuencia_documental.items()
    if df > 0
}

filas_solicitadas = []
for termino in terminos_ejemplo:
    df = frecuencia_documental.get(termino, 0)
    filas_solicitadas.append({
        "término": termino,
        "df(t)": df,
        "idf solicitado": idf_solicitado.get(termino)
    })

pd.DataFrame(filas_solicitadas)


,término,df(t),idf solicitado
0,the,100,0.693147
1,and,100,0.693147
2,book,86,0.771399
3,project,100,0.693147
4,gutenberg,100,0.693147


### Variante suavizada alternativa con `+1` dentro del denominador

Una variante también frecuente en recuperación de información es:

\[
ln\left(
rac{N}{df(t)+1}
ight)
\]

No se confunde con la expresión solicitada anteriormente. Su cálculo se incluye como comparación adicional.

In [7]:
idf_denominador_suavizado = {
    termino: math.log(N / (df + 1))
    for termino, df in frecuencia_documental.items()
}

pd.DataFrame({
    "término": terminos_ejemplo,
    "df(t)": [frecuencia_documental.get(t, 0) for t in terminos_ejemplo],
    "ln(N/df+1) solicitado": [idf_solicitado.get(t) for t in terminos_ejemplo],
    "ln(N/(df+1))": [idf_denominador_suavizado.get(t) for t in terminos_ejemplo]
})


,término,df(t),ln(N/df+1) solicitado,ln(N/(df+1))
0,the,100,0.693147,-0.009950
1,and,100,0.693147,-0.009950
2,book,86,0.771399,0.139262
3,project,100,0.693147,-0.009950
4,gutenberg,100,0.693147,-0.009950


## 6. TF-IDF con `TfidfVectorizer`

Primero se calcula sin suavizado. Al no utilizar la configuración por defecto de scikit-learn, se calcula TF-IDF mediante `TfidfVectorizer` sobre los documentos normalizados.

In [8]:
vectorizador_clasico = TfidfVectorizer(
    lowercase=False,
    tokenizer=normalizar_y_tokenizar,
    token_pattern=None,
    smooth_idf=False,
    norm="l2"
)

matriz_tfidf_clasica = vectorizador_clasico.fit_transform(documentos)
vocabulario = vectorizador_clasico.vocabulary_

print("Forma de la matriz TF-IDF clásica:", matriz_tfidf_clasica.shape)

for termino in terminos_ejemplo:
    if termino in vocabulario:
        print(
            f"idf de '{termino}' según TfidfVectorizer:",
            vectorizador_clasico.idf_[vocabulario[termino]]
        )


Forma de la matriz TF-IDF clásica: (100, 103964)
idf de 'the' según TfidfVectorizer: 1.0
idf de 'and' según TfidfVectorizer: 1.0
idf de 'book' según TfidfVectorizer: 1.1508228897345836
idf de 'project' según TfidfVectorizer: 1.0
idf de 'gutenberg' según TfidfVectorizer: 1.0


## 7. `TfidfVectorizer` con suavizado

El suavizado predeterminado de scikit-learn utiliza:

\[
idf(t)=\ln\left(
rac{N+1}{df(t)+1}
ight)+1
\]

Además del IDF, se obtiene la matriz TF-IDF completa de los 100 documentos.

In [9]:
vectorizador_suavizado = TfidfVectorizer(
    lowercase=False,
    tokenizer=normalizar_y_tokenizar,
    token_pattern=None,
    smooth_idf=True,
    norm="l2"
)

matriz_tfidf_suavizada = vectorizador_suavizado.fit_transform(documentos)
vocabulario_suavizado = vectorizador_suavizado.vocabulary_

print("Forma de la matriz TF-IDF suavizada:", matriz_tfidf_suavizada.shape)

for termino in terminos_ejemplo:
    if termino in vocabulario_suavizado:
        print(
            f"idf suavizado de '{termino}':",
            vectorizador_suavizado.idf_[vocabulario_suavizado[termino]]
        )


Forma de la matriz TF-IDF suavizada: (100, 103964)
idf suavizado de 'the': 1.0
idf suavizado de 'and': 1.0
idf suavizado de 'book': 1.1492123981866758
idf suavizado de 'project': 1.0
idf suavizado de 'gutenberg': 1.0


## 8. Comparación de los métodos de IDF

Se construye una tabla para términos presentes en el corpus. Cada columna corresponde a una definición distinta del IDF.

In [10]:
comparacion = []

for termino in terminos_ejemplo:
    df = frecuencia_documental.get(termino, 0)
    if df == 0:
        continue

    indice = vocabulario_suavizado[termino]
    comparacion.append({
        "término": termino,
        "df(t)": df,
        "idf clásico manual": idf_clasico[termino],
        "idf solicitado ln(N/df+1)": idf_solicitado[termino],
        "idf sklearn suavizado": vectorizador_suavizado.idf_[indice]
    })

tabla_comparacion = pd.DataFrame(comparacion)
tabla_comparacion


,término,df(t),idf clásico manual,idf solicitado ln(N/df+1),idf sklearn suavizado
0,the,100,0.000000,0.693147,1.000000
1,and,100,0.000000,0.693147,1.000000
2,book,86,0.150823,0.771399,1.149212
3,project,100,0.000000,0.693147,1.000000
4,gutenberg,100,0.000000,0.693147,1.000000


## 9. Ver el TF-IDF de un documento

Los valores se ordenan para observar cuáles términos tienen mayor peso TF-IDF dentro del primer documento.

In [11]:
indice_documento = 0
fila = matriz_tfidf_suavizada[indice_documento]

valores = pd.DataFrame({
    "término": [vectorizador_suavizado.get_feature_names_out()[i] for i in fila.indices],
    "tf-idf": fila.data
}).sort_values("tf-idf", ascending=False)

print("Documento:", nombres_documentos[indice_documento])
valores.head(20)


Documento: pg10000.txt


,término,tf-idf
0,the,0.557120
4,of,0.439268
16,and,0.327259
49,to,0.238140
34,or,0.177753
13,in,0.167526
227,shall,0.159734
228,be,0.139767
94,our,0.126131
132,barons,0.111521


## Conclusión

Se procesaron exactamente los primeros 100 archivos `.txt` del corpus. Se verificó la frecuencia documental manual y se calcularon dos fórmulas de IDF, además de los resultados de `TfidfVectorizer` con y sin suavizado.

Las escalas numéricas pueden diferir según la presencia de `+1` o su ubicación en la fórmula, pero todas las variantes buscan reducir el peso de los términos que aparecen en muchos documentos.